Exercice 1 : Chargement et préparation des données


Dans cet exercice, nous allons configurer l'environnement et préparer le jeu de données que nous utiliserons tout au long de ce projet. Une préparation adéquate des données garantit le bon déroulement des étapes suivantes, telles que la génération d'embeddings, l'utilisation de bases de données vectorielles ou la construction de modèles d'apprentissage automatique. Examinons chaque étape ensemble.



Pourquoi cette étape est importante :
Avant de se pencher sur les techniques avancées, il est crucial de :

Assurez-vous que toutes les bibliothèques requises sont installées.
Chargez et examinez les données pour comprendre leur structure.
Préparez un sous-ensemble gérable pour des itérations plus rapides pendant le développement.
Ces étapes nous aident à éviter les problèmes techniques et à garantir que nos analyses ou modèles reposent sur des bases solides.

In [ ]:
import numpy as np
import pandas as pd
!pip install faiss-cpu -qq
!pip install chromadb -qq
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [ ]:
path = "/content/sample_data/california_housing_train.csv"
pdf = pd.read_csv(path)

In [ ]:
pdf["id"] = range(len(pdf))  # IDs numériques uniques : 0, 1, 2, ...

In [ ]:
display(pdf)

# Analyses complémentaires utiles
print(pdf.shape)           # Dimensions : (nb_lignes, nb_colonnes)
print(pdf.dtypes)          # Types des colonnes
print(pdf.isnull().sum())  # Valeurs manquantes par colonne

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,id
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0,0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0,1
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0,2
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0,3
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0,4
...,...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0,16995
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0,16996
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0,16997
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0,16998


(17000, 10)
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
id                      int64
dtype: object
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
id                    0
dtype: int64


In [ ]:
pdf_subset = pdf.head(1000).reset_index(drop=True)
print(f"Sous-ensemble : {pdf_subset.shape}")  # → (1000, n_colonnes)

Sous-ensemble : (1000, 10)


In [ ]:
# Chargement
path = "/content/sample_data/california_housing_train.csv"
pdf = pd.read_csv(path)

# Identifiant unique
pdf["id"] = range(len(pdf))

# Exploration
display(pdf)
print(pdf.shape)
print(pdf.isnull().sum())

# Sous-ensemble de développement
pdf_subset = pdf.head(1000).reset_index(drop=True)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,id
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0,0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0,1
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0,2
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0,3
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0,4
...,...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0,16995
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0,16996
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0,16997
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0,16998


(17000, 10)
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
id                    0
dtype: int64


Exercice 2 : Vectorisation avec des transformateurs de phrases


Dans cet exercice, nous allons transformer nos données textuelles (titres d'articles) en représentations numériques appelées plongements lexicaux . Cette étape est cruciale pour permettre aux machines de comprendre et de traiter des données textuelles dans des tâches telles que la recherche de similarités, le clustering et l'apprentissage automatique. Nous utiliserons Sentence Transformers , une bibliothèque populaire pour générer des représentations vectorielles denses de texte.



Pourquoi cette étape est importante :

Les machines ne peuvent pas traiter directement du texte brut ; elles ont besoin d’entrées numériques. Les vecteurs d’intégration (embeddings) sont des vecteurs denses qui capturent le sens et le contexte du texte. En générant des vecteurs d’intégration pour nos titres d’articles, nous les rendons utilisables pour des tâches en aval telles que la recherche de similarités ou l’alimentation de modèles d’apprentissage automatique.



In [ ]:
from sentence_transformers import SentenceTransformer, InputExample

# Sous-ensemble créé à l'exercice 1
pdf_subset = pdf.head(1000).reset_index(drop=True)

In [ ]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    return InputExample(texts=[doc1])  # Le titre est passé comme texte d'entrée

In [ ]:
faiss_train_examples = pdf_subset.apply(
    lambda x: example_create_fn(str(x["median_house_value"])), axis=1
).tolist()

faiss_train_examples[:10]  # Vérification des 10 premiers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
# Modèle léger (~80MB), 384 dimensions, excellent rapport qualité/vitesse

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
titles_list = pdf_subset["median_house_value"].astype(str).tolist()
print(f"Nombre de valeurs: {len(titles_list)}")
print(f"Exemple : {titles_list[0]}")

Nombre de valeurs: 1000
Exemple : 66900.0


In [ ]:
faiss_title_embedding = model.encode(
    titles_list,
    show_progress_bar=True  # Optionnel mais utile pour suivre la progression
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
len(faiss_title_embedding), len(faiss_title_embedding[0])
# Attendu → (1000, 384)
# 1000 embeddings (un par titre), chaque vecteur de dimension 384

(1000, 384)

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample

# Sous-ensemble
pdf_subset = pdf.head(1000).reset_index(drop=True)

# Fonction auxiliaire
def example_create_fn(doc1: pd.Series) -> InputExample:
    return InputExample(texts=[doc1])

# Préparation des InputExamples
faiss_train_examples = pdf_subset.apply(
    lambda x: example_create_fn(str(x["median_house_value"])), axis=1
).tolist()

# Modèle
model = SentenceTransformer("all-MiniLM-L6-v2")

# Titres bruts
titles_list = pdf_subset["median_house_value"].astype(str).tolist()

# Génération des embeddings
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)

# Vérification
print(len(faiss_title_embedding), len(faiss_title_embedding[0]))
# → (1000, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

1000 384


Exercice 3 : Indexation et recherche FAISS


Dans cet exercice, nous utiliserons FAISS (Facebook AI Similarity Search) pour créer un index des plongements lexicaux générés lors de l'exercice précédent. Cela nous permettra d'effectuer des recherches de similarité rapides et efficaces sur de vastes ensembles de vecteurs. L'objectif est de permettre la récupération des articles d'actualité les plus pertinents en fonction de la requête d'un utilisateur.



Pourquoi cette étape est importante :


FAISS est une bibliothèque conçue pour effectuer des recherches de similarité à grande échelle. La recherche efficace au sein d'embeddings (vecteurs de grande dimension) devient complexe lorsqu'on travaille avec ces derniers. FAISS propose des algorithmes optimisés d'indexation et de recherche, permettant de retrouver des éléments similaires en quelques millisecondes, même dans de vastes ensembles de données.

In [ ]:
import numpy as np
import faiss

In [ ]:
pdf_to_index = pdf_subset.copy()                          # Le sous-ensemble de l'exercice 1
id_index = np.array(pdf_to_index["id"].tolist())          # Tableau d'IDs uniques (int64 requis par FAISS)

In [ ]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
# IndexFlatIP  → produit scalaire (= cosinus sur vecteurs normalisés)
# IndexIDMap   → associe chaque vecteur à son ID original

# Normalisation des embeddings pour la similarité cosinus
faiss.normalize_L2(faiss_title_embedding)

index_content.add_with_ids(faiss_title_embedding, id_index)

print(f"Vecteurs indexés : {index_content.ntotal}")  # → 1000

Vecteurs indexés : 1000


In [ ]:
def search_content(query, pdf_to_index, k=3):
    # Encoder la requête
    query_vector = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vector)                     # Normaliser la requête

    # Recherche dans l'index
    top_k = index_content.search(query_vector, k)        # Retourne (distances, ids)
    ids = top_k[1][0]                                    # IDs des k voisins les plus proches
    similarities = top_k[0][0]                           # Scores de similarité correspondants

    # Récupérer les articles depuis le DataFrame
    results = pdf_to_index[pdf_to_index["id"].isin(ids)].copy()
    results["similarities"] = similarities               # Ajouter les scores

    return results

In [ ]:
display(search_content("animal", pdf_to_index, k=5))

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,id,similarities
107,-115.69,32.79,18.0,1564.0,340.0,1161.0,343.0,2.1792,55200.0,107,0.255401
424,-116.95,32.76,13.0,5543.0,857.0,2074.0,737.0,4.9528,266200.0,424,0.241975
482,-116.98,32.75,18.0,1519.0,369.0,802.0,347.0,2.9886,170800.0,482,0.225565
565,-117.01,32.67,17.0,2319.0,348.0,1125.0,337.0,5.5510,266900.0,565,0.223466
632,-117.03,32.67,15.0,5094.0,818.0,2118.0,758.0,5.3505,266600.0,632,0.219206


In [ ]:
import numpy as np
import faiss

# 2. Préparation
pdf_to_index = pdf_subset.copy()
id_index = np.array(pdf_to_index["id"].tolist())

# 3. Normalisation
content_encoded_normalized = np.array(faiss_title_embedding).astype("float32")
faiss.normalize_L2(content_encoded_normalized)

# 4. Index FAISS
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)
print(f"Vecteurs indexés : {index_content.ntotal}")

# 5. Fonction de recherche
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vector)
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0]
    similarities = top_k[0][0]
    results = pdf_to_index[pdf_to_index["id"].isin(ids)].copy()
    results["similarities"] = similarities
    return results

# 6. Test
display(search_content("animal", pdf_to_index, k=5))

Vecteurs indexés : 1000


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,id,similarities
107,-115.69,32.79,18.0,1564.0,340.0,1161.0,343.0,2.1792,55200.0,107,0.255401
424,-116.95,32.76,13.0,5543.0,857.0,2074.0,737.0,4.9528,266200.0,424,0.241975
482,-116.98,32.75,18.0,1519.0,369.0,802.0,347.0,2.9886,170800.0,482,0.225565
565,-117.01,32.67,17.0,2319.0,348.0,1125.0,337.0,5.5510,266900.0,565,0.223466
632,-117.03,32.67,15.0,5094.0,818.0,2118.0,758.0,5.3505,266600.0,632,0.219206


Exercice 4 : Collection et requêtes ChromaDB


Dans cet exercice, nous présenterons ChromaDB , une base de données vectorielle open source conçue pour stocker, indexer et interroger des vecteurs d'embeddings. ChromaDB simplifie la manipulation des embeddings et, contrairement à FAISS, gère automatiquement la tokenisation, l'embedding et l'indexation sans nécessiter de génération manuelle d'embeddings. Elle est donc idéale pour l'intégration avec des applications basées sur des modèles de langage étendus (LLM ), notamment pour la création de systèmes de questions-réponses ou de moteurs de recherche.



Pourquoi cette étape est importante :

Une fois les représentations vectorielles de nos données générées, l'étape suivante consiste à les stocker et à les interroger efficacement. ChromaDB offre une interface de haut niveau pour la gestion de ces représentations et prend en charge les métadonnées, ce qui en fait une solution idéale pour développer des applications telles que la recherche documentaire ou les systèmes de questions-réponses. À l'aide de ChromaDB, nous montrons comment intégrer ces représentations vectorielles dans un flux de travail concret permettant d'interroger et de récupérer des documents pertinents.



Instructions:

In [ ]:
import chromadb
from chromadb.config import Settings
import json

In [ ]:
chroma_client = chromadb.Client()
collection_name = "my_news"

# Supprimer la collection si elle existe déjà
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

Creating collection: 'my_news'


In [ ]:
display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()]  # IDs uniques en string (requis par ChromaDB)
)

print(f"Documents ajoutés : {collection.count()}")  # → 100

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,id
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0,0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0,1
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0,2
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0,3
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0,4
...,...,...,...,...,...,...,...,...,...,...
995,-117.09,32.55,8.0,6533.0,1217.0,4797.0,1177.0,3.9583,144400.0,995
996,-117.10,34.57,6.0,5110.0,1044.0,1938.0,724.0,3.1917,112800.0,996
997,-117.10,34.21,22.0,4397.0,931.0,1145.0,445.0,4.5268,108400.0,997
998,-117.10,34.03,24.0,4144.0,826.0,2127.0,772.0,2.5172,96000.0,998


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.6MiB/s]


Documents ajoutés : 100


In [81]:
results = collection.query(
    query_texts=["space"],   # ChromaDB génère l'embedding automatiquement
    n_results=10             # Les 10 documents les plus proches sémantiquement
)

print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "80",
            "25",
            "24",
            "40",
            "87",
            "5",
            "31",
            "55",
            "71",
            "89"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "62500.0",
            "100000.0",
            "53500.0",
            "64000.0",
            "63500.0",
            "74000.0",
            "67500.0",
            "67500.0",
            "67500.0",
            "67500.0"
        ]
    ],
    "uris": null,
    "included": [
        "metadatas",
        "documents",
        "distances"
    ],
    "data": null,
    "metadatas": [
        [
            {
                "housing_age": 20.0
            },
            {
                "housing_age": 34.0
            },
            {
                "housing_age": 18.0
            },
            {
                "housing_age": 17.0
            },
            {
                "housing_age": 17.0
            },
   

In [84]:
import chromadb
from chromadb.config import Settings
import json

# 2. Client et collection
chroma_client = chromadb.Client()
collection_name = "my_news"

if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

# 3. Ajout des données
display(pdf_subset)

collection.add(
    documents=pdf_subset["median_house_value"][:100].astype(str).tolist(),
    metadatas=[{"housing_age": topic} for topic in pdf_subset["housing_median_age"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()]
)
print(f"Documents ajoutés : {collection.count()}")

# 4. Requête
results = collection.query(
    query_texts=["space"],
    n_results=10
)
print(json.dumps(results, indent=4))

Creating collection: 'my_news'


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,id
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0,0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0,1
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0,2
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0,3
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0,4
...,...,...,...,...,...,...,...,...,...,...
995,-117.09,32.55,8.0,6533.0,1217.0,4797.0,1177.0,3.9583,144400.0,995
996,-117.10,34.57,6.0,5110.0,1044.0,1938.0,724.0,3.1917,112800.0,996
997,-117.10,34.21,22.0,4397.0,931.0,1145.0,445.0,4.5268,108400.0,997
998,-117.10,34.03,24.0,4144.0,826.0,2127.0,772.0,2.5172,96000.0,998


Documents ajoutés : 100
{
    "ids": [
        [
            "80",
            "25",
            "24",
            "40",
            "87",
            "5",
            "31",
            "55",
            "71",
            "89"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "62500.0",
            "100000.0",
            "53500.0",
            "64000.0",
            "63500.0",
            "74000.0",
            "67500.0",
            "67500.0",
            "67500.0",
            "67500.0"
        ]
    ],
    "uris": null,
    "included": [
        "metadatas",
        "documents",
        "distances"
    ],
    "data": null,
    "metadatas": [
        [
            {
                "housing_age": 20.0
            },
            {
                "housing_age": 34.0
            },
            {
                "housing_age": 18.0
            },
            {
                "housing_age": 17.0
            },
            {
                "housing_age":

Exercice 5 : Réponse aux questions avec un modèle de visage faisant un câlin


Dans cet exercice, nous allons mettre en pratique tous les éléments en construisant un système de questions-réponses (Q/R) utilisant le modèle de langage Hugging Face . En combinant la recherche documentaire (via ChromaDB) avec la génération de texte (via Hugging Face), nous créons un pipeline simple mais puissant où un modèle génère des réponses en fonction du contexte pertinent.



Pourquoi cette étape est importante :


La récupération des documents pertinents ne représente que la moitié du travail. L'étape suivante consiste à générer des réponses pertinentes à partir de ces contenus. Il s'agit d'une technique fondamentale des systèmes modernes de génération augmentée par la récupération (RAG) , où un modèle de langage exploite à la fois des connaissances pré-entraînées et des informations externes pour répondre aux questions avec plus de précision. En intégrant ChromaDB et les transformateurs Hugging Face , nous simulons un processus de questions-réponses réaliste.



In [85]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Modèle
model_id  = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model  = AutoModelForCausalLM.from_pretrained(model_id)

# Pipeline
pipe = pipeline("text-generation", model=lm_model,
                tokenizer=tokenizer, max_new_tokens=512, device_map="auto")

# Prompt
question        = "What's the latest news on space development?"
context         = " ".join([f"#{str(i)}" for i in results["documents"][0]])
prompt_template = f"Relevant context: {context}\n\nThe user's question: {question}"

# Génération
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=

Relevant context: #62500.0 #100000.0 #53500.0 #64000.0 #63500.0 #74000.0 #67500.0 #67500.0 #67500.0 #67500.0

The user's question: What's the latest news on space development?

The most important topic for a question like this is the number of users (or their personal space of choice) who are using Space. It's a lot of fun to answer and interesting to read about as well.

The following question is the most relevant for the user: What's the latest news about space development?

The most important topic for a question like this is the number of users (or their personal space of choice) who are using Space. It's a lot of fun to answer and interesting to read about as well.

The question itself is interesting, but it doesn't necessarily mean it's a bad question. It's just about the current situation. The question itself is interesting, but it doesn't necessarily mean it's a bad question. It's just about the current situation.

The question itself is interesting, but it doesn't necessarily 